# Global 모델링 Stage 0 — 데이터·split 확인

이 Notebook은 `baseline_42features`의 실제 Global Dataset과 고정 SAMPID split을 확인한다. 모델을 학습·선택하거나 결과를 해석하지 않는다.

재사용 기능은 `code.pipeline.saved_results`와 `code.evaluation.data_checks`에서 import한다.

In [ ]:
import os
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path(os.environ.get('KHUDA_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'code').is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError('KHUDA_PROJECT_ROOT에 저장소 루트를 지정하거나 저장소 안에서 Notebook을 실행하세요.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# IPython이 미리 불러온 표준 라이브러리 code와 저장소 패키지 code의 이름 충돌을 해소한다.
if 'code' in sys.modules and not hasattr(sys.modules['code'], '__path__'):
    del sys.modules['code']

from code.evaluation import summarize_global_modeling_inputs
from code.pipeline.saved_results import load_saved_global_train_test
from code.preprocess.build_features import feature_columns, load_feature_config

EXPERIMENT = 'baseline_42features'
RESULT_ROOT = ROOT / 'data' / 'result' / EXPERIMENT
FEATURE_CONFIG = ROOT / 'code' / 'config' / 'features.yaml'
DATASET_PATH = RESULT_ROOT / 'datasets' / 'global_dataset.parquet'
SPLIT_PATH = RESULT_ROOT / 'splits' / 'split_ids.csv'

for path in (DATASET_PATH, SPLIT_PATH):
    if not path.exists():
        raise FileNotFoundError(f'필요한 Stage 0 입력 파일이 없습니다: {path}')

PALETTE = {'blue': '#0066cc', 'ink': '#1d1d1f', 'parchment': '#f5f5f7', 'hairline': '#e0e0e0'}
plt.rcParams.update({'figure.facecolor': PALETTE['parchment'], 'axes.facecolor': '#ffffff', 'axes.edgecolor': PALETTE['hairline'], 'text.color': PALETTE['ink']})

In [ ]:
feature_config = load_feature_config(FEATURE_CONFIG)
expected_features = feature_columns(feature_config)
train_bundle, test_bundle = load_saved_global_train_test(DATASET_PATH, SPLIT_PATH, feature_config)
report = summarize_global_modeling_inputs(train_bundle, test_bundle, expected_features=expected_features)

print(f'Experiment: {EXPERIMENT}')
print(f'Expected feature count: {len(expected_features)}')
display(report['sample_summary'])
display(report['split_checks'])
display(report['target_summary'])
display(report['baseline_summary'])
display(report['feature_presence'])

## 시각화

아래 그래프는 확인용이다. 결과의 의미나 다음 단계 진행 여부는 사람이 직접 판단·기록한다.

In [ ]:
# 1. Train/Test Target 분포
target_plot = report['target_summary'].pivot(index='split', columns='target', values='count').fillna(0)
target_plot.columns = ['y=0', 'y=1']
ax = target_plot.plot.bar(color=[PALETTE['ink'], PALETTE['blue']], figsize=(7, 4), rot=0)
ax.set_title('Train/Test Target count')
ax.set_xlabel('')
ax.set_ylabel('Person-Period count')
ax.legend(title='employment_transition')
plt.tight_layout()
plt.show()

In [ ]:
# 2. baseline_year별 취업전환율
baseline_plot = report['baseline_summary'].pivot(index='baseline_year', columns='split', values='employment_transition_rate')
ax = baseline_plot.plot.bar(color=[PALETTE['ink'], PALETTE['blue']], figsize=(7, 4), rot=0)
ax.set_title('Employment transition rate by baseline year')
ax.set_xlabel('baseline_year')
ax.set_ylabel('Target rate (y=1)')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Feature 결측률 상위 항목
missingness = report['missingness'].pivot(index='feature', columns='split', values='missing_rate').fillna(0)
top_missing = missingness.loc[missingness.max(axis=1).sort_values(ascending=False).head(15).index].sort_values(by=list(missingness.columns))
ax = top_missing.plot.barh(color=[PALETTE['ink'], PALETTE['blue']], figsize=(8, 6))
ax.set_title('Top 15 feature missingness rates')
ax.set_xlabel('Missing rate')
ax.set_ylabel('Feature')
ax.set_xlim(0, 1)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

display(top_missing)

## 다음 단계 전 사람 확인

- 이 Notebook은 사실 확인만 제공한다. 표본·분할·Feature·결측률의 연구적 의미는 여기에서 해석하지 않는다.
- Stage 1로 넘어갈지 여부와 그 근거는 담당자가 직접 기록한다.